# 🛸 Orbit Wars — Laboratorio de Iteraciones

Notebook completo para desarrollar, probar y comparar versiones del bot
desde Google Colab.

**Flujo de trabajo:**
1. Instalar dependencias
2. Clonar repositorio
3. Cargar y probar el bot actual
4. Visualizar partidas
5. Comparar versiones
6. Preparar y descargar submission


## ⚙️ 1. Instalación de dependencias

In [ ]:
# Instalar el entorno oficial de Kaggle
!pip install kaggle-environments --quiet
print('✅ kaggle-environments instalado')

## 📂 2. Clonar repositorio

In [ ]:
import os

REPO_URL = 'https://github.com/juandalfonso/orbit-wars-bot.git'
REPO_DIR = 'orbit-wars-bot'

if os.path.exists(REPO_DIR):
    print('🔄 Actualizando repositorio...')
    !cd {REPO_DIR} && git pull
else:
    print('📥 Clonando repositorio...')
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f'✅ Directorio actual: {os.getcwd()}')

## 🤖 3. Cargar el bot

In [ ]:
import importlib.util, sys

def load_bot(version='v1'):
    path = f'src/bots/{version}_main.py'
    spec = importlib.util.spec_from_file_location('bot', path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    print(f'✅ Bot {version} cargado desde {path}')
    return mod.agent

agent_v1 = load_bot('v1')

## 🎮 4. Correr una partida de prueba

In [ ]:
from kaggle_environments import make

env = make('orbit_wars', debug=True)
env.run([agent_v1, 'random'])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f'Jugador {i}: reward={s.reward}, status={s.status}')

## 📺 5. Visualizar la partida

In [ ]:
# Renderiza la partida en el notebook
env.render(mode='ipython', width=700, height=700)

## 📊 6. Torneo de prueba: bot vs random (N partidas)

In [ ]:
def run_tournament(agent_fn, opponent='random', n_games=10, verbose=True):
    wins, losses, errors = 0, 0, 0
    rewards = []
    for i in range(n_games):
        try:
            env = make('orbit_wars', debug=False)
            env.run([agent_fn, opponent])
            final = env.steps[-1]
            my_reward = final[0].reward
            opp_reward = final[1].reward
            rewards.append(my_reward or 0)
            if my_reward is not None and (opp_reward is None or my_reward > opp_reward):
                wins += 1
                if verbose: print(f'  Partida {i+1}: ✅ GANA (bot={my_reward:.0f}, opp={opp_reward:.0f})')
            else:
                losses += 1
                if verbose: print(f'  Partida {i+1}: ❌ PIERDE (bot={my_reward:.0f}, opp={opp_reward:.0f})')
        except Exception as e:
            errors += 1
            if verbose: print(f'  Partida {i+1}: ⚠️ ERROR — {e}')
    avg = sum(rewards) / len(rewards) if rewards else 0
    print(f'\n🏆 Resultado: {wins}W / {losses}L / {errors}E de {n_games} partidas')
    print(f'📊 Reward promedio: {avg:.1f}')
    return {'wins': wins, 'losses': losses, 'errors': errors, 'avg_reward': avg}

print('🎮 Torneo v1 vs random (10 partidas)...')
stats_v1 = run_tournament(agent_v1, n_games=10)

## 🧪 7. Área de experimentación — Escribe tu nueva versión aquí

In [ ]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

# 👇 ESCRIBE TU NUEVA VERSIÓN DEL BOT AQUÍ
# Puedes modificar cualquier función y experimentar
# sin tocar los archivos del repositorio

def agent_experimental(obs):
    '''
    Versión experimental — modifica esta función para probar ideas.
    Copia el código de v1 y modifica desde ahí.
    '''
    # Por ahora, usa la misma lógica de v1 como base
    return agent_v1(obs)


## 🥊 8. Comparar versión experimental vs v1

In [ ]:
def head_to_head(agent_a, agent_b, n_games=10, label_a='A', label_b='B'):
    '''Enfrenta dos bots entre sí y muestra resultados.'''
    wins_a, wins_b, draws = 0, 0, 0
    for i in range(n_games):
        env = make('orbit_wars', debug=False)
        env.run([agent_a, agent_b])
        final = env.steps[-1]
        ra = final[0].reward or 0
        rb = final[1].reward or 0
        if ra > rb: wins_a += 1
        elif rb > ra: wins_b += 1
        else: draws += 1
    print(f'\n🥊 {label_a} vs {label_b} ({n_games} partidas):')
    print(f'  {label_a}: {wins_a} victorias')
    print(f'  {label_b}: {wins_b} victorias')
    print(f'  Empates: {draws}')
    return wins_a, wins_b, draws

# Cambiar 'experimental' cuando tengas una segunda versión lista
head_to_head(agent_experimental, agent_v1, n_games=10,
             label_a='Experimental', label_b='v1')

## 📤 9. Preparar archivo de submission

In [ ]:
import shutil, os
from google.colab import files

def prepare_submission(version='v1'):
    '''
    Copia la versión indicada como main.py y la descarga.
    Este es el archivo que se sube a Kaggle.
    '''
    src = f'src/bots/{version}_main.py'
    dst = 'main.py'
    shutil.copy(src, dst)
    print(f'✅ main.py listo (copia de {version}_main.py)')
    print(f'📦 Descargando...')
    files.download(dst)

prepare_submission('v1')
# Cuando tengas v2 lista: prepare_submission('v2')

## 📝 10. Bitácora del experimento de hoy

In [ ]:
bitacora = {
    'version': 'v1',
    'fecha': '2026-04-21',
    'hipotesis': 'Expansión segura con scoring por producción y reserva defensiva',
    'cambios': [
        'target_score() pondera producción, distancia, costo, sol',
        'ships_available() con reserva mínima configurable',
        'Prefiere neutrales sobre enemigos',
    ],
    'resultados': stats_v1,
    'notas': 'Completar después de revisar replays',
}

import json
print(json.dumps(bitacora, indent=2))